### load patches

In [ ]:
import numpy as np, zarr, json, os
from pathlib import Path
from scipy import ndimage as ndi
from skimage.segmentation import watershed

TRAIN_ITER = "v1"
OVERWRITE = True

VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # used in baseline (v0)
    "6bba_61dd1e0d", # added in first iter (v1)
]
MAX_PATCHES = {
    "6bba_61dd1e0d": 100, # added in first iter (v1)
}

raw, lab = {}, {}
for p in VIDEOS:
    r = np.asarray(zarr.open(f'unet_patches/{p}.zarr/0', mode="r"))
    l = np.load(f'unet_labels/{p}_pts.npy', allow_pickle=True)
    n = MAX_PATCHES.get(p)
    if n is not None:
        r = r[:n]
        l = l[l[:, 0] < n]                         
    raw[p], lab[p] = r, l
    print(p, raw[p].shape, lab[p].shape)

44b6_33b596bf (50, 16, 64, 64) (678, 4)
44b6_40c45f5a (200, 16, 64, 64) (3782, 4)
44b6_551a5dba (228, 16, 64, 64) (6196, 4)
6bba_61dd1e0d (100, 16, 64, 64) (231, 4)


### Gaussian heatmaps

In [2]:
from scipy import ndimage as ndi

SIGMA = (1.0, 4.0, 4.0)   # (z, y, x) voxels; set from cell size
zz, yy, xx = np.ogrid[:16, :64, :64]

def pts_for_patch(arr, i):
    if arr.dtype == object:                          # one array of points per patch
        return np.asarray(arr[i], float).reshape(-1, 3)
    if arr.ndim == 2 and arr.shape[1] == 4:          # rows of (patch_idx, z, y, x)
        return arr[arr[:, 0] == i, 1:].astype(float)
    raise ValueError(f"unrecognised points layout: {arr.shape}")

def make_heatmap(pts):
    hm = np.zeros((16, 64, 64), np.float32)
    for cz, cy, cx in pts:
        g = np.exp(-(((zz-cz)/SIGMA[0])**2 + ((yy-cy)/SIGMA[1])**2 + ((xx-cx)/SIGMA[2])**2) / 2)
        hm = np.maximum(hm, g.astype(np.float32))
    return hm

heat = {}
for name in VIDEOS:
    out = Path("heatmaps") / f"{name}_{TRAIN_ITER}_heatmap.npy"
    
    if not OVERWRITE and out.exists():
        heat[name] = np.load(out); print("loaded existing:", out); continue
    
    out.parent.mkdir(parents=True, exist_ok=True)
    N = raw[name].shape[0]
    heat[name] = np.stack([make_heatmap(pts_for_patch(lab[name], i)) for i in range(N)])
    np.save(out, heat[name])
    
    print(name, "generated", heat[name].shape, "patches with no points:",
          int((heat[name].reshape(N, -1).max(1) == 0).sum()))

44b6_33b596bf generated (50, 16, 64, 64) patches with no points: 0
44b6_40c45f5a generated (200, 16, 64, 64) patches with no points: 0
44b6_551a5dba generated (228, 16, 64, 64) patches with no points: 0
6bba_61dd1e0d generated (100, 16, 64, 64) patches with no points: 0


### preprocessing

In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader

norm_path = f"models/norm_{TRAIN_ITER}.json"

if not OVERWRITE and os.path.exists(norm_path):
    with open(norm_path, "r") as f:
        norm_params = json.load(f)
    lo, hi = norm_params["lo"], norm_params["hi"]
    print("loaded existing norm:", norm_path)
else:
    allraw = np.concatenate([raw[n].ravel() for n in VIDEOS])
    lo, hi = np.percentile(allraw, [1, 99.8])

    with open(norm_path, "w") as f:
        json.dump({"lo": float(lo), "hi": float(hi)}, f)

    print("computed and saved norm:", norm_path)

print("norm range:", lo, hi)

def norm(x):
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

X = np.concatenate([norm(raw[n]) for n in VIDEOS])[:, None]
Y = np.concatenate([heat[n] for n in VIDEOS])[:, None]

print(X.shape, Y.shape, "target max:", Y.max())

loader = DataLoader(
    TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
    batch_size=8,
    shuffle=True,
    drop_last=False
)

computed and saved norm: models/norm_v1.json
norm range: 0.0 3614.0
(578, 1, 16, 64, 64) (578, 1, 16, 64, 64) target max: 1.0


### 3D U-Net

In [ ]:
%%writefile unet3d.py
import torch, torch.nn as nn

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True),
        nn.Conv3d(o, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True)
        )

class UNet3D(nn.Module):
    # anisotropic pooling: z is only 16 slices, so don't squash it early
    def __init__(self, c=16):
        super().__init__()
        self.e1, self.e2, self.e3, self.b = block(1, c), block(c, 2*c), block(2*c, 4*c), block(4*c, 8*c)
        self.p1, self.p2, self.p3 = nn.MaxPool3d((1,2,2)), nn.MaxPool3d(2), nn.MaxPool3d(2)
        self.u3 = nn.ConvTranspose3d(8*c, 4*c, 2, stride=2)
        self.u2 = nn.ConvTranspose3d(4*c, 2*c, 2, stride=2)
        self.u1 = nn.ConvTranspose3d(2*c, c, (1,2,2), stride=(1,2,2))
        self.d3, self.d2, self.d1 = block(8*c, 4*c), block(4*c, 2*c), block(2*c, c)
        self.out = nn.Conv3d(c, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

Overwriting unet3d.py


In [5]:
from unet3d import UNet3D
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet3D(16).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(device, sum(p.numel() for p in model.parameters())/1e6, "M params")

cuda 1.399921 M params


### training

In [6]:
EPOCHS, SAVE_EVERY = 100, 10

def weighted_mse(pred, target):
    w = 1 + 20 * target          # upweight cell voxels, background is the vast majority
    return (w * (pred - target) ** 2).mean()

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)   
        opt.zero_grad()
        loss = weighted_mse(model(xb), yb)
        loss.backward(); opt.step()
        tot += loss.item() * len(xb)
    print(f"epoch {ep:3d}  loss {tot/len(X):.5f}")
    if ep % SAVE_EVERY == 0:
        torch.save(model.state_dict(), f"models/unet_{TRAIN_ITER}_e{ep}.pt")

epoch   1  loss 0.18670
epoch   2  loss 0.09383
epoch   3  loss 0.05421
epoch   4  loss 0.03494
epoch   5  loss 0.02661
epoch   6  loss 0.02089
epoch   7  loss 0.01751
epoch   8  loss 0.01619
epoch   9  loss 0.01329
epoch  10  loss 0.01155
epoch  11  loss 0.01090
epoch  12  loss 0.00944
epoch  13  loss 0.00807
epoch  14  loss 0.00807
epoch  15  loss 0.00847
epoch  16  loss 0.00609
epoch  17  loss 0.00534
epoch  18  loss 0.00616
epoch  19  loss 0.00559
epoch  20  loss 0.00438
epoch  21  loss 0.00400
epoch  22  loss 0.00413
epoch  23  loss 0.00394
epoch  24  loss 0.00314
epoch  25  loss 0.00391
epoch  26  loss 0.00341
epoch  27  loss 0.00319
epoch  28  loss 0.00257
epoch  29  loss 0.00250
epoch  30  loss 0.00241
epoch  31  loss 0.00423
epoch  32  loss 0.00364
epoch  33  loss 0.00276
epoch  34  loss 0.00231
epoch  35  loss 0.00243
epoch  36  loss 0.00210
epoch  37  loss 0.00249
epoch  38  loss 0.00279
epoch  39  loss 0.00255
epoch  40  loss 0.00213
epoch  41  loss 0.00191
epoch  42  loss 